In [1]:
pip install gymnasium minigrid stable-baselines3 gym_minigrid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.7/136.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 9.0 MB/s eta 0:00:00
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827727 sha256=c2a14b982da1b1e36feb87c230419b4bf7599a9bc76af29009a744cf3085f035
  Stored in directory: /root/.cache/pip/wheels/95/51/6c/9bb05ebbe7c5cb8171dfaa3611f32622ca4658d53f31c79077
Successfully built gym
  Attempting uninstall: gym
    Found existing installation: gym 0.25.2
    Uninstalling gym-0.25.2:
      Successfully uninstalled gym-0.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

# MiniGrid Kitchen Environment — RL with PPO

This notebook walks through a complete reinforcement learning pipeline for a **custom MiniGrid kitchen environment**. The agent's task is to clean dirty dishes by:

1. **Picking up** dirty dishes (red balls) from the kitchen grid
2. **Rinsing** them at the sink (top-left corner) using the toggle action
3. **Loading** them into the dishwasher (bottom-right corner) using the toggle action

We use **Proximal Policy Optimization (PPO)** from Stable-Baselines3 with a custom CNN feature extractor to train the agent from pixel observations.

### Outline
1. Imports & Configuration
2. Custom Kitchen Environment Definition
3. Environment Registration
4. CNN Feature Extractor
5. Environment Factory
6. Training
7. Evaluation
8. Video Recording

## 1. Imports & Configuration

We import the following key libraries:

- **`gymnasium`** — the standard RL environment interface (successor to OpenAI Gym)
- **`minigrid`** — a suite of lightweight, grid-based environments perfect for RL research
- **`stable_baselines3`** — high-quality implementations of RL algorithms (PPO, A2C, DQN, etc.)
- **`torch`** — PyTorch, used under the hood by SB3 and for defining our custom CNN
- **`imageio`** — for saving rollout frames as video files

We also set a global seed and environment ID for reproducibility.

In [2]:
import gymnasium as gym
import minigrid
from minigrid.wrappers import ImgObsWrapper

import torch as th
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage

import imageio
from pathlib import Path
import numpy as np
import random
from typing import List, Tuple

CURRENT_DIR = Path(".")  # notebook runs from its own directory
ENV_ID = "MiniGrid-Kitchen-v0"
SEED = 42

/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### MiniGrid Internal Imports

We need access to MiniGrid's internal building blocks to create our custom environment:

| Class | Purpose |
|---|---|
| `MiniGridEnv` | Base class for all MiniGrid environments |
| `Grid` | The 2D grid that holds all objects |
| `Ball` | Used here to represent dirty dishes (red balls) |
| `Goal` | Used to represent the dishwasher location |
| `Wall` | Obstacles / countertops in the kitchen |

The import path varies between `minigrid` and `gym_minigrid` depending on the installed version, so we try both.

In [11]:
try:
    # Modern minigrid (gymnasium compatible) - for version 3.0+
    from minigrid.core.grid import Grid
    from minigrid.core.world_object import Ball, Goal, Wall
    from minigrid.minigrid_env import MiniGridEnv
except ImportError:
    # Fallback for older versions
    try:
        from minigrid.minigrid import MiniGridEnv, Grid, Ball, Goal, Wall
    except ImportError:
        from gym_minigrid.minigrid import MiniGridEnv, Grid, Ball, Goal, Wall

## 2. Custom Kitchen Environment

The `KitchenMiniGridEnv` extends MiniGrid's base environment with a custom kitchen scenario.

### Grid Layout
- **5×5 grid** surrounded by walls
- **Sink** at position `(1, 1)` — top-left interior
- **Dishwasher** (Goal tile) at position `(3, 3)` — bottom-right interior
- **1–3 dirty dishes** (red Balls) placed randomly
- **1 random wall** placed as a counter obstacle

### Reward Structure

| Event | Reward |
|---|---|
| Every step | **-0.01** (encourages efficiency) |
| Rinsing a dish at the sink | **+0.2** (intermediate reward for progress) |
| Loading a rinsed dish into dishwasher | **+1.0** (main objective reward) |
| Tight looping (optional) | **-0.5** (anti-loop penalty when enabled) |

### Dish Cleaning Logic
The agent must follow a specific sequence:
1. Navigate to a dish and **pick it up** (pickup action)
2. Navigate to the sink at `(1,1)` and **toggle** → marks the dish as "rinsed"
3. Navigate to the dishwasher at `(width-2, height-2)` and **toggle** → dish is cleaned, reward granted

### Anti-Loop Detection
When `anti_loop_penalty=True`, the environment tracks the agent's last 20 positions. If the last 6 positions only contain ≤ 2 unique locations, a penalty of -0.5 is applied to discourage the agent from getting stuck in repetitive movement patterns.

### Termination
The episode ends when:
- All dishes have been cleaned (no balls remain on the grid or in the agent's inventory), or
- The maximum number of steps (`max_steps=100`) is reached

In [60]:
import gymnasium.utils.seeding

class KitchenMiniGridEnv(MiniGridEnv):
    """
    MiniGrid environment representing a tiny kitchen:
    - Dirty dishes are represented as red Ball objects.
    - Sink is a coordinate where agent can 'use' to rinse (we track rinsed status).
    - Dishwasher is a Goal tile: dropping a rinsed dish on the dishwasher counts as 'cleaned'.
    """

    def __init__(
        self,
        size: int = 8,
        num_dishes: Tuple[int, int] = (1, 3),
        max_steps: int = 100,
        seed: int = None,
        anti_loop_penalty: bool = False,
        render_mode: str = "rgb_array",
    ):
        self.kitchen_size = size
        self.num_dishes_range = num_dishes
        self.anti_loop_penalty = anti_loop_penalty

        # Define the mission space (required by newer minigrid versions)
        mission_space = gym.spaces.Text(max_length=256)

        super().__init__(
            mission_space=mission_space,
            width=size, height=size, max_steps=max_steps, see_through_walls=True,
            render_mode=render_mode
        )
        self.seed(seed)
        self.rinsed_ids = set()
        self.cleaned_count = 0
        self.dish_ids = []
        self._last_positions = []

    def seed(self, seed=None):
        """Explicitly define seed method for compatibility."""
        self.np_random, seed = gymnasium.utils.seeding.np_random(seed)
        return [seed]

    def _gen_grid(self, width, height):
        """Generate the kitchen grid layout."""
        # Create an empty grid with walls around the border
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        # Place a random internal wall (kitchen counter)
        for i in range(1):
            wx = self._rand_int(1, width - 1)
            wy = self._rand_int(1, height - 1)
            if self.grid.get(wx, wy) is None:
                self.grid.set(wx, wy, Wall())

        # Place dishwasher (Goal) at bottom-right interior
        dx = width - 2
        dy = height - 2
        self.put_obj(Goal(), dx, dy)
        self.dishwasher_pos = (dx, dy)

        # Sink location at top-left interior
        sx = 1
        sy = 1
        self.sink_pos = (sx, sy)

        # Place 1-3 dirty dishes (red Ball objects) at random free cells
        num_dishes = self._rand_int(self.num_dishes_range[0], self.num_dishes_range[1] + 1)
        self.dish_ids = []
        placed = 0
        while placed < num_dishes:
            x = self._rand_int(1, width - 1)
            y = self._rand_int(1, height - 1)
            if self.grid.get(x, y) is None and (x, y) != self.dishwasher_pos and (x, y) != self.sink_pos:
                ball = Ball(color="red")
                self.put_obj(ball, x, y)
                self.dish_ids.append(ball.cur_pos)
                placed += 1

        # Place the agent at a random free cell
        self.place_agent()
        self.mission = "Clean the dishes: rinse at sink then drop them on dishwasher."

    def step(self, action):
        """
        Extend step with custom reward shaping for the dish-cleaning task.
        """
        obs, reward, terminated, truncated, info = super().step(action)

        # Step penalty to encourage efficiency
        reward = reward - 0.01

        # Track positions for loop detection
        pos = tuple(self.agent_pos)
        self._last_positions.append(pos)
        if len(self._last_positions) > 20:
            self._last_positions.pop(0)

        carried = getattr(self, "carrying", None)

        # Dense reward for carrying a dish (Encourages pickup)
        if carried is not None and carried.type == "ball":
            reward += 0.05

        # Check if agent is at dishwasher position
        ax, ay = self.agent_pos
        if (ax, ay) == self.dishwasher_pos:
            obj = self.grid.get(ax, ay)
            if carried is None:
                pass

        # Toggle action handles rinsing and loading
        if action == self.actions.toggle:
            # Rinsing: carrying a ball at the sink
            if pos == self.sink_pos and carried is not None and carried.type == "ball":
                carried.rinsed = True
                carried.color = "blue"  # VISUALIZATION: Turn dish blue when rinsed
                reward += 0.5 # Increased from 0.2

            # Loading: carrying a rinsed ball at the dishwasher
            if pos == self.dishwasher_pos and carried is not None and getattr(carried, "rinsed", False):
                reward += 1.0
                self.cleaned_count += 1
                self.carrying = None

        # Anti-loop penalty
        if self.anti_loop_penalty:
            if len(self._last_positions) >= 12:
                if len(set(self._last_positions[-6:])) <= 2:
                    reward -= 0.5

        # Check termination: are all dishes cleaned?
        remaining_balls = 0
        for i in range(1, self.width - 1):
            for j in range(1, self.height - 1):
                o = self.grid.get(i, j)
                if o is not None and o.type == "ball":
                    remaining_balls += 1
        if getattr(self, "carrying", None) is not None and self.carrying.type == "ball":
            remaining_balls += 1

        if remaining_balls == 0:
            terminated = True

        return obs, reward, terminated, truncated, info

## 3. Environment Registration

We register the custom environment with Gymnasium's registry so it can be instantiated via `gym.make("MiniGrid-Kitchen-v0", ...)`. This is the standard pattern for creating reusable custom environments.

The `try/except` block handles the case where the environment is already registered (e.g., if this cell is re-run).

In [61]:
from gymnasium.envs.registration import register

try:
    register(
        id=ENV_ID,
        entry_point=lambda **kwargs: KitchenMiniGridEnv(**kwargs),
        max_episode_steps=100,
    )
except Exception:
    pass  # Already registered

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment MiniGrid-Kitchen-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


## 4. CNN Feature Extractor

Since our observations are **pixel images** of the grid (thanks to `ImgObsWrapper`), we need a convolutional neural network to extract meaningful features before feeding them into the PPO policy and value networks.

### Architecture

```
Input Image (C, H, W)
    │
    ├── Conv2d(C → 32, 3×3, padding=1) + ReLU
    ├── Conv2d(32 → 64, 3×3, padding=1) + ReLU
    ├── Conv2d(64 → 64, 3×3, padding=1) + ReLU
    ├── Flatten
    └── Linear(flattened → 128) + ReLU
         │
         └── 128-dim feature vector
```

Key design choices:
- **3×3 kernels with padding=1** preserve spatial dimensions, which is important for small grid images
- **No pooling layers** — the MiniGrid observations are already very small (e.g., 7×7×3), so we don't want to downsample further
- The flattened CNN output is projected to a **128-dimensional** feature vector that PPO's policy and value heads consume

In [62]:
class MinigridFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        # Dynamically compute the flattened size by passing a dummy input
        with th.no_grad():
            sample = observation_space.sample()[None]
            sample = th.as_tensor(sample).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations):
        return self.linear(self.cnn(observations))


policy_kwargs = dict(
    features_extractor_class=MinigridFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

## 5. Environment Factory

The `make_env` function is a **factory** that returns a callable which creates a properly wrapped environment. This pattern is required by SB3's vectorized environment wrappers.

The wrapping chain is:
1. `gym.make(...)` → creates the raw `KitchenMiniGridEnv`
2. `ImgObsWrapper(env)` → extracts only the image observation (discarding the mission string and direction), returning `(H, W, C)` arrays
3. `DummyVecEnv([make_env()])` → wraps it in SB3's vectorized env interface (batch dimension)
4. `VecTransposeImage(vec_env)` → transposes images from `(H, W, C)` to `(C, H, W)` as PyTorch expects

In [75]:
def make_env(seed=SEED, size=8, anti_loop_penalty=False):
    """Returns a callable that creates a wrapped MiniGrid kitchen environment."""
    def _init():
        env = gym.make(
            ENV_ID,
            size=size,
            num_dishes=(1, 3),
            max_steps=100,
            seed=seed,
            anti_loop_penalty=anti_loop_penalty,
            render_mode="rgb_array" # Ensure render_mode is passed
        )
        env = ImgObsWrapper(env)
        return env

    return _init

## 6. Training

We train a **PPO (Proximal Policy Optimization)** agent — one of the most popular and reliable on-policy RL algorithms.

### PPO Hyperparameters

| Parameter | Value | Description |
|---|---|---|
| `learning_rate` | 3e-4 | Adam optimizer learning rate |
| `n_steps` | 1024 | Number of steps per rollout before each update |
| `batch_size` | 64 | Minibatch size for each gradient step |
| `n_epochs` | 10 | Number of passes over the rollout buffer per update |
| `gamma` | 0.99 | Discount factor — how much the agent values future rewards |
| `gae_lambda` | 0.95 | GAE (Generalized Advantage Estimation) smoothing factor |
| `clip_range` | 0.2 | PPO clipping range — limits how much the policy can change per update |
| `ent_coef` | 0.02 | Entropy bonus — encourages exploration |
| `vf_coef` | 0.5 | Value function loss coefficient |
| `max_grad_norm` | 0.5 | Gradient clipping to prevent exploding gradients |

### Training Pipeline
1. Set random seeds for reproducibility
2. Create a vectorized environment with image transposition
3. Instantiate PPO with the custom CNN feature extractor
4. Train for 200,000 timesteps (adjustable)
5. Save the trained model to disk

In [76]:
# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)

# Create vectorized environment
# TRAIN ON 8x8 GRID with ANTI-LOOP PENALTY
vec_env = DummyVecEnv([make_env(seed=SEED, size=8, anti_loop_penalty=False)])
vec_env = VecTransposeImage(vec_env)

# Initialize PPO with custom CNN policy
model = PPO(
    "CnnPolicy",
    vec_env,
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.05,  # INCREASED: Encourage more exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
    seed=SEED,
    verbose=1,
    tensorboard_log=str(CURRENT_DIR / "logs"),
)

# Train the agent
print("Starting training on 8x8 grid with Anti-Loop Penalty (500k steps)...")
model.learn(total_timesteps=500_000, progress_bar=True)  # INCREASED: More time to learn

# Save the trained model
model_path = CURRENT_DIR / "model"
model.save(model_path)
print(f"Model saved to {model_path}.zip")

Using cuda device
Starting training on 8x8 grid with Anti-Loop Penalty (500k steps)...
Logging to logs/PPO_8


Output()

/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs 
returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")

/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


Streaming output truncated to the last 5000 lines.
|    time_elapsed         | 714        |
|    total_timesteps      | 215040     |
| train/                  |            |
|    approx_kl            | 0.01826552 |
|    clip_fraction        | 0.171      |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.63      |
|    explained_variance   | 0.706      |
|    learning_rate        | 0.0003     |
|    loss                 | 2.2        |
|    n_updates            | 2090       |
|    policy_gradient_loss | -0.0148    |
|    value_loss           | 5.09       |
----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 300         |
|    iterations           | 211         |
|    time_elapsed         | 718         |
|    total_timesteps      | 216064      |
| train/                  |             |
|    approx_kl            | 0.013318882 |
|    clip_fraction        | 0.158      

Model saved to model.zip


## 7. Evaluation

After training, we evaluate the agent's performance over multiple episodes using SB3's `evaluate_policy` utility. This runs the trained policy in the environment for `n_eval_episodes` episodes and computes the **mean** and **standard deviation** of the total episode rewards.

A higher mean reward indicates that the agent has learned to:
- Efficiently pick up dishes
- Rinse them at the sink
- Load them into the dishwasher
- Minimize wasted steps

We use a different seed (`SEED + 1`) for evaluation to test generalization to unseen random configurations.

In [77]:
eval_env = make_env(seed=SEED + 1, size=8, anti_loop_penalty=False)()

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"\nEvaluation results (20 episodes): Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

eval_env.close()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")



Evaluation results (20 episodes): Mean reward: -0.51 ± 1.46


## 8. Video Recording

Finally, we record a video of the trained agent performing a single episode. This is useful for:
- **Visual debugging** — seeing if the agent actually follows the rinse → load sequence
- **Presentations** — demonstrating the learned behavior
- **Sanity checks** — verifying the environment renders correctly

The video is saved as `demo.mp4` at 6 FPS. Each frame is captured using `env.render(mode="rgb_array")`.

In [78]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Manually create environment with render_mode="rgb_array" for recording
record_env = gym.make(
    ENV_ID,
    size=8,  # Increased size as requested
    num_dishes=(1, 3),
    max_steps=200,
    max_episode_steps=200,  # Override the registered TimeLimit of 100
    seed=SEED + 2,
    anti_loop_penalty=False,
    render_mode="rgb_array"
)
record_env = ImgObsWrapper(record_env)

frames: List[np.ndarray] = []
obs, _ = record_env.reset(seed=SEED + 2)

def annotate_frame(env, frame, step_idx, action_name=None):
    """Overlay text on the frame to explain what's happening."""
    img = Image.fromarray(frame)
    draw = ImageDraw.Draw(img)

    # Status info
    carried = getattr(env.unwrapped, "carrying", None)
    status = "Idling"
    if carried:
        status = f"Carrying {carried.color} dish"
        if getattr(carried, "rinsed", False):
            status += " (Rinsed)"

    text = f"Step: {step_idx}\nStatus: {status}"
    if action_name:
        text += f"\nAction: {action_name}"

    # Draw text (top-left)
    # We rely on default font since custom fonts might not be available
    draw.text((5, 5), text, fill=(255, 255, 255))
    return np.array(img)

# Initial frame
frame = record_env.render()
frames.append(annotate_frame(record_env, frame, 0))

done = False
steps = 0
max_steps = 200

while not done and steps < max_steps:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = record_env.step(action)

    # Get action name
    action_name = record_env.unwrapped.actions(action).name

    frame = record_env.render()
    annotated_frame = annotate_frame(record_env, frame, steps + 1, action_name)
    frames.append(annotated_frame)

    done = terminated or truncated
    steps += 1

output_path = CURRENT_DIR / "demo.mp4"
imageio.mimsave(output_path, frames, fps=6)
print(f"Video saved to {output_path} ({len(frames)} frames, {steps} steps)")

record_env.close()

Video saved to demo.mp4 (201 frames, 200 steps)


---

### Next Steps

- **Enable anti-loop penalty**: Set `anti_loop_penalty=True` in `make_env()` to discourage repetitive agent behavior
- **Increase training timesteps**: Try 500k or 1M steps for better convergence
- **Tune hyperparameters**: Experiment with `ent_coef`, `learning_rate`, or `n_steps`
- **Scale the kitchen**: Increase `size` and `num_dishes` for a harder task
- **Monitor with TensorBoard**: Run `tensorboard --logdir logs/` to visualize training curves